In [1]:
from glmsingle.glmsingle import GLM_single
import argparse
import os
import os.path as op
from nilearn import image
from stress_risk.utils.data import Subject
from nilearn.glm.first_level import make_first_level_design_matrix
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
subject = '01'
session = 1


derivatives = op.join(bids_folder, 'derivatives')

runs = range(1, 7)

sub = Subject(subject, bids_folder=bids_folder)

ims = sub.get_preprocessed_bold(session=session)

base_dir = 'glm_stim1.denoise'

In [3]:

data = [image.load_img(im).get_fdata() for im in ims]

base_dir = op.join(derivatives, base_dir, f'sub-{subject}',
                    f'ses-{session}', 'func')

if not op.exists(base_dir):
    os.makedirs(base_dir)

onsets = sub.get_fmri_events(session=session, runs = runs)
tr = 2.3
n = 135
frametimes = np.linspace(tr/2., (n - .5)*tr, n)
onsets['onset'] = ((onsets['onset']+tr/2.) // 2.3) * 2.3

print(onsets)

dm = [make_first_level_design_matrix(frametimes, onsets.loc[run], hrf_model='fir', oversampling=100.,
                                        drift_order=0,
                                        drift_model=None).drop('constant', axis=1) for run in runs]

dm = pd.concat(dm, keys=range(1, 7), names=['run']).fillna(0)
dm.columns = [c.replace('_delay_0', '') for c in dm.columns]
dm /= dm.max()
#print(dm)
dm[dm < 1.0] = 0.0
print(dm.shape)

X = [dm.loc[run].values for run in runs]

print(len(X))

     onset  trial_nr    trial_type  duration    n2
run                                               
1     13.8         1  trial_001_n1       0.6   NaN
1     34.5         2          n2_7       0.6   7.0
1     50.6         3          n2_7       0.6   7.0
1     64.4         4         n2_28       0.6  28.0
1     75.9         5         n2_10       0.6  10.0
..     ...       ...           ...       ...   ...
6    230.0       116  trial_116_n1       0.6   NaN
6    243.8       117  trial_117_n1       0.6   NaN
6    257.6       118  trial_118_n1       0.6   NaN
6    140.3       110  trial_110_n1       0.6   NaN
6    299.0       120         n2_32       0.6  32.0

[240 rows x 5 columns]
(810, 142)
6


In [9]:
onsets = sub.get_fmri_events(session=session, runs = runs)
print(np.shape(onsets))
onsets

(240, 5)


,onset,trial_nr,trial_type,duration,n2
run,,,,,
1,13.262186,1,trial_001_n1,0.6,NaN
1,35.115593,2,n2_7,0.6,7.0
1,51.263774,3,n2_7,0.6,7.0
1,64.409383,4,n2_28,0.6,28.0
1,76.570633,5,n2_10,0.6,10.0
...,...,...,...,...,...
6,230.529428,116,trial_116_n1,0.6,NaN
6,244.175394,117,trial_117_n1,0.6,NaN
6,258.338458,118,trial_118_n1,0.6,NaN


In [4]:
opt = dict()
opt['wantlibrary'] = 1
opt['wantglmdenoise'] = 1
opt['wantfracridge'] = 1
opt['wantfileoutputs'] = [0, 0, 0, 1]

In [ ]:
glmsingle_obj = GLM_single(opt)

results_glmsingle = glmsingle_obj.fit(
    X,
    data,
    0.6,
    2.3,
    outputdir=base_dir)


In [ ]:

betas = results_glmsingle['typed']['betasmd']
betas = image.new_img_like(ims[0], betas)
betas = image.index_img(betas, slice(None, None, 2))
betas.to_filename(op.join(base_dir, f'sub-{subject}_ses-{session}_task-risk_space-T1w_desc-stims1_pe.nii.gz'))


In [6]:
np.shape(data)

(6, 194480, 135)

In [7]:
np.shape(X)

(6, 135, 142)

In [8]:
print(dm.shape)

(810, 142)


In [12]:
X[1][X[1] != 0.]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1.])

In [13]:
np.shape(image.load_img(ims[0]).get_fdata())

(55, 68, 52, 135)